
# Homework 1 — Logical Inference with SymPy & pyDatalog

**Course:** Artificial Intelligence  
**Names:**  
**Deliverables:** Jupyter Notebook  
**This version intentionally omits most implementations. Your task is to complete all TODOs.**



## 0. Environment & Setup

Required packages (install in a terminal):

```bash
pip install sympy pyDatalog
```


In [2]:
!/usr/local/bin/python3.10 -m pip install sympy pyDatalog


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [3]:

# Imports (do not change names; the autograder expects these)
import time, re
from typing import List, Set, Tuple, Dict
from sympy import Symbol
from sympy.logic.boolalg import And, Or, Not, Implies, to_cnf
from sympy.logic.inference import satisfiable
from pyDatalog import pyDatalog as _pyd

print("Environment OK")


Environment OK



## 1. Knowledge Base Format and Parsing (TODO)

Input file: `premises.txt` — Lines are either facts or Horn clauses.

- **Fact:** `fact: a`  
- **Rule:** `a & b -> c` (left: 0+ premises; right: 1 conclusion)  
- Comments start with `#`  
- Symbol regex: `^[a-z][a-z0-9_]*$`

**Tasks**
1. Implement `parse_premises(path)` to return `(facts, rules)`
2. Write a brief validation (e.g., assert that symbols match the regex)


In [4]:

# === TODO: Implement the parser ===
sym_re = re.compile(r'^[a-z][a-z0-9_]*$')

def parse_premises(path: str) -> Tuple[Set[str], List[Tuple[Set[str], str]]]:
    """
    Reads the KB file and returns:
      facts: set[str]
      rules: list[tuple[set[str], str]] as (premises, conclusion)
    - Accepts lines like 'fact: rain' or 'a & b -> c'
    - Ignore comments and blank lines
    - Validate symbols using `sym_re`
    """
    

    lines_fact = []
    lines_rules = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f.readlines():
            line = line.strip()
            if not line or line.startswith("#"):
                # Comment or blank line
                continue
            elif line.startswith("fact:"):
                lines_fact.append(line)
            else:
                lines_rules.append(line)
                
    facts = set()
    rules = list()

    for fact in lines_fact:
        fact_name = fact.split(":")[-1].strip()
        if not sym_re.fullmatch(fact_name):
            raise ValueError(f"Invalid fact symbol: {fact_name}")
        facts.add(fact_name)

    for rule in lines_rules:
        rule_parts = rule.split("->")
        rule_premise_part = rule_parts[0].strip()
        rule_premises = {p.strip() for p in rule_premise_part.split("&")}
        rule_conclusion = rule_parts[1].strip()

        for sym in rule_premises | {rule_conclusion}:
            if not sym_re.fullmatch(sym):
                raise ValueError(f"Invalid symbol in rule: {sym}")

        rules.append((rule_premises, rule_conclusion))
    
                            
    return facts, rules
    #raise NotImplementedError("parse_premises is not implemented yet.")

# Path (you may change if needed)
PREMISES_PATH = 'premises_python.txt'
facts, rules = parse_premises(PREMISES_PATH)
print(f"facts={facts}\nrules={rules}")


facts={'transport_available', 'have_seeds', 'rain', 'storage_ready', 'sunlight'}
rules=[({'rain'}, 'wet_ground'), ({'wet_ground', 'have_seeds'}, 'germination'), ({'germination', 'sunlight'}, 'grow'), ({'grow'}, 'harvest_possible'), ({'grow', 'sunlight'}, 'healthy_crops'), ({'healthy_crops', 'storage_ready'}, 'stored_goods'), ({'transport_available', 'stored_goods'}, 'logistics_ok'), ({'harvest_possible', 'logistics_ok'}, 'profit')]



## 2. SymPy Modeling (TODO)

Build a propositional model of the KB and implement entailment via **refutation**.


In [5]:

# === TODO: Build symtab and KB expression ===
def build_symbol_table(facts: Set[str], rules: List[Tuple[Set[str], str]]) -> Dict[str, Symbol]:
    """Return a mapping {name: Symbol(name)} covering all identifiers in facts and rules."""

    names = set(facts)
    for premises, conclusion in rules:
        names.update(premises)
        names.update({conclusion})

    return {name: Symbol(name) for name in sorted(names)}
    #return names

    raise NotImplementedError

def kb_to_sympy(symtab: Dict[str, Symbol], facts: Set[str], rules: List[Tuple[Set[str], str]]):
    """Return a SymPy Boolean expression for the entire KB.
       Encode facts as unit clauses, rules as implications (premises -> conclusion).
    """
    clauses = []

    for fact in facts: 
        clauses.append(Implies(True, symtab[fact]))
        #clauses.append(symtab[fact])

    for premises, conclusion in rules:
        prem_res = True
        if(len(premises) == 0): # Assumption: {() -> X} ==> X = True
            prem_res = True
        else:
            for p in premises:
                prem_res = And(prem_res, symtab[p])

        clauses.append(Implies(prem_res, symtab[conclusion]))
    
    return And(*clauses)
    raise NotImplementedError

def entails_sympy(kb_expr, query_sym: Symbol) -> bool:
    """Return True iff KB ⊨ query (via refutation: KB ∧ ¬query is UNSAT)."""
    print("kb_expr: ", kb_expr)
    print("query_sym: ", query_sym)
    print("Not(query_sym): ", Not(query_sym))
    res = satisfiable(And(kb_expr, Not(query_sym)))
    return res is False
    raise NotImplementedError

# Example usage (uncomment after you implement):
facts, rules = parse_premises(PREMISES_PATH)
symtab = build_symbol_table(facts, rules)
print(symtab)
kb_expr = kb_to_sympy(symtab, facts, rules)
print("kb_expr: ", kb_expr)
print("-"*20)
print("Entails 'harvest_possible'?", entails_sympy(kb_expr, symtab['harvest_possible']))
print("-"*20)


{'germination': germination, 'grow': grow, 'harvest_possible': harvest_possible, 'have_seeds': have_seeds, 'healthy_crops': healthy_crops, 'logistics_ok': logistics_ok, 'profit': profit, 'rain': rain, 'storage_ready': storage_ready, 'stored_goods': stored_goods, 'sunlight': sunlight, 'transport_available': transport_available, 'wet_ground': wet_ground}
kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & storage_ready, stored_goods)) & (Implies(stored_goods & transport_available, logistics_ok))
--------------------
kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunli

In [6]:
# AND of ORs
cnf_expr = to_cnf(kb_expr, force=True)
print("KB (CNF): ", cnf_expr)
print("KB (CNF): ", str(cnf_expr).split("&"))

KB (CNF):  have_seeds & rain & storage_ready & sunlight & transport_available & (harvest_possible | ~grow) & (wet_ground | ~rain) & (germination | ~have_seeds | ~wet_ground) & (grow | ~germination | ~sunlight) & (healthy_crops | ~grow | ~sunlight) & (logistics_ok | ~stored_goods | ~transport_available) & (profit | ~harvest_possible | ~logistics_ok) & (stored_goods | ~healthy_crops | ~storage_ready)
KB (CNF):  ['have_seeds ', ' rain ', ' storage_ready ', ' sunlight ', ' transport_available ', ' (harvest_possible | ~grow) ', ' (wet_ground | ~rain) ', ' (germination | ~have_seeds | ~wet_ground) ', ' (grow | ~germination | ~sunlight) ', ' (healthy_crops | ~grow | ~sunlight) ', ' (logistics_ok | ~stored_goods | ~transport_available) ', ' (profit | ~harvest_possible | ~logistics_ok) ', ' (stored_goods | ~healthy_crops | ~storage_ready)']



## 3. pyDatalog Forward-Chaining (TODO)

Recreate the same KB in **pyDatalog** (0-arity predicates) and check if the target facts hold.


In [7]:

# === TODO: Translate the KB into pyDatalog ===
def datalog_reset():
    _pyd.clear()

def datalog_create_predicates(names: Set[str]):
    """Create 0-arity predicates dynamically for each name."""
    terms_string = ', '.join(sorted(names))
    _pyd.create_terms(terms_string)
    #raise NotImplementedError

def datalog_assert_kb(facts: Set[str], rules: List[Tuple[Set[str], str]]):
    """Assert facts and rules into pyDatalog.
       a & b -> c   becomes   c <= (a & b)
       fact: a      becomes   +a()
    """

    print(facts)
    base = []
    for fact in facts:
        base.append(f"+{fact}()")

    for premises, conclusion in rules:
        if len(premises) == 0:
            base.append(f"+{conclusion}()")
        
        else:
            prems_str = " & ".join(f"{prem}()" for prem in premises)
            base.append(f"{conclusion}() <= ({prems_str})")

    program = '\n'.join(base)
    _pyd.load(program)

            



    
    #raise NotImplementedError

def datalog_holds(atom_name: str) -> bool:
    """Return True iff the 0-arity predicate 'atom_name()' has a solution in pyDatalog."""
    return _pyd.ask(f"{atom_name}()") is not None

    raise NotImplementedError

# Example usage (uncomment after you implement):
datalog_reset()
datalog_create_predicates(set(symtab.keys()))
datalog_assert_kb(facts, rules)
print("harvest_possible():", datalog_holds('harvest_possible'))


{'transport_available', 'have_seeds', 'rain', 'storage_ready', 'sunlight'}
harvest_possible(): True



## 4. Consistency & Performance (TODO)

Using the same query name (e.g., `'harvest_possible'`):

1. Verify that **SymPy entailment** and **pyDatalog** agree.  
2. Measure and print wall-clock times with `time.time()`.


In [8]:

# === TODO: Run consistency and timing ===
query_name = 'harvest_possible'
t0 = time.time()
ent_sympy = entails_sympy(kb_expr, symtab[query_name])
t1 = time.time()
t2 = time.time()
ent_datalog = datalog_holds(query_name)
t3 = time.time()
print(f"SymPy: {ent_sympy} time={t1-t0:.6f}s | pyDatalog: {ent_datalog} time={t3-t2:.6f}s")


kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & storage_ready, stored_goods)) & (Implies(stored_goods & transport_available, logistics_ok))
query_sym:  harvest_possible
Not(query_sym):  ~harvest_possible
SymPy: True time=0.001165s | pyDatalog: True time=0.001894s


In [9]:

# === TODO: Run consistency and timing ===
query_names = ['harvest_possible', 'healthy_crops', 'logistics_ok', 'profit']
for query_name in query_names:
    t0 = time.time()
    ent_sympy = entails_sympy(kb_expr, symtab[query_name])
    t1 = time.time()
    t2 = time.time()
    ent_datalog = datalog_holds(query_name)
    t3 = time.time()
    print(f"SymPy: {ent_sympy} time={t1-t0:.6f}s | pyDatalog: {ent_datalog} time={t3-t2:.6f}s")
    print("---"*20)


kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & storage_ready, stored_goods)) & (Implies(stored_goods & transport_available, logistics_ok))
query_sym:  harvest_possible
Not(query_sym):  ~harvest_possible
SymPy: True time=0.001498s | pyDatalog: True time=0.000197s
------------------------------------------------------------
kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & stora

In [10]:
other_query_names = ['rain', 'have_seeds', 'sunlight', 'storage_ready', 'transport_available', 'wet_ground',
                     'germination', 'grow', 'harvest_possible', 'healthy_crops', 'stored_goods', 'logistics_ok',
                     'profit']
for query_name in other_query_names:
    t0 = time.time()
    ent_sympy = entails_sympy(kb_expr, symtab[query_name])
    t1 = time.time()
    t2 = time.time()
    ent_datalog = datalog_holds(query_name)
    t3 = time.time()
    print(f"SymPy: {ent_sympy} time={t1-t0:.6f}s | pyDatalog: {ent_datalog} time={t3-t2:.6f}s")
    print("---"*20)

kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & storage_ready, stored_goods)) & (Implies(stored_goods & transport_available, logistics_ok))
query_sym:  rain
Not(query_sym):  ~rain
SymPy: True time=0.001638s | pyDatalog: True time=0.000174s
------------------------------------------------------------
kb_expr:  have_seeds & rain & storage_ready & sunlight & transport_available & (Implies(grow, harvest_possible)) & (Implies(rain, wet_ground)) & (Implies(germination & sunlight, grow)) & (Implies(grow & sunlight, healthy_crops)) & (Implies(harvest_possible & logistics_ok, profit)) & (Implies(have_seeds & wet_ground, germination)) & (Implies(healthy_crops & storage_ready, stored_goods))


## 5. Short Report

Discuss the following in the next cell:
- Modeling choices & assumptions 
- Correctness agreement (SymPy vs. pyDatalog)  
- Runtime observations 
- Limitations of each approach

=== TODO: Report for findings ===

- Modeling choices & assumptions 
    - Because pyDatalog has syntax limitations, both systems were tested on a model consisting only of individual expressions such as facts and AND-based rules. It was assumed that there would be no complex rules in the premises text file. The sample input given for the assignment was also prepared accordingly.
    
- Correctness agreement (SymPy vs. pyDatalog)  
    - I compare the outputs with using suggested queries, and all the outputs came same. 
    - I also checked the results of each component and they all come out True, looking at the Facts and Rules in the txt file it seems that this is what should happen.
- Runtime observations 
    - pyDatalog is much faster than symPy. The logic behind pyDatalog is specialized for evaluating the Horn Clauses, its a forward chaining engine with recursiveness. symPy holds logical expressions symbolically and evaluate with some kind of satisfiability search operations, that will make system complicated and slow.


- Limitations of each approach
    - pyDatalog can only work with 'and' operations (Horn Clauses). 
    - symPy's limitation is its high computational cost.



## 6. Submission Checklist

- [ ] `parse_premises` implemented and passing your tests  
- [ ] SymPy: symbol table, KB expression, entailment function implemented  
- [ ] pyDatalog: predicates created, KB asserted, query checks implemented  
- [ ] Consistency & timing comparison executed  
- [ ] Clean code & brief report
